# Traffic Sign Image Prediction

Upload one PNG, JPG, or JPEG traffic sign image to test the already-trained CNN.

In [ ]:
from io import BytesIO
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from IPython.display import clear_output, display
from PIL import Image

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
MODEL_PATH = PROJECT_ROOT / "models" / "traffic_sign_cnn.keras"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Trained model was not found: {MODEL_PATH.resolve()}")

model = tf.keras.models.load_model(MODEL_PATH)
print(f"Loaded model: {MODEL_PATH.resolve()}")

## Upload a traffic sign image

Choose one PNG, JPG, or JPEG image. The image is processed in memory and is not saved or added to the dataset.

In [ ]:
upload_widget = widgets.FileUpload(
    accept=".png,.jpg,.jpeg",
    multiple=False,
    description="Choose Files",
)
prediction_output = widgets.Output()

def get_uploaded_file(upload_value):
    if isinstance(upload_value, dict):
        return next(iter(upload_value.values()))
    return upload_value[0]

def predict_uploaded_image(change):
    if not upload_widget.value:
        return

    uploaded_file = get_uploaded_file(upload_widget.value)
    image_bytes = uploaded_file["content"]

    with prediction_output:
        clear_output(wait=True)

        with Image.open(BytesIO(image_bytes)) as uploaded_image:
            original_image = uploaded_image.copy()
            processed_image = uploaded_image.convert("RGB").resize((32, 32), Image.Resampling.LANCZOS)

        image_array = np.asarray(processed_image, dtype=np.float32) / 255.0
        image_batch = np.expand_dims(image_array, axis=0)
        prediction_probabilities = model.predict(image_batch, verbose=0)
        predicted_class = int(np.argmax(prediction_probabilities[0]))
        confidence = float(np.max(prediction_probabilities[0]))

        figure, axis = plt.subplots(figsize=(4, 4))
        axis.imshow(original_image)
        axis.axis("off")
        axis.set_title("Uploaded image")
        plt.show()

        print(f"Predicted class: {predicted_class}")
        print(f"Confidence: {confidence:.2%}")

upload_widget.observe(predict_uploaded_image, names="value")
display(widgets.VBox([upload_widget, prediction_output]))

The notebook uses the existing model only. It does not retrain, modify, save over, or replace the model, and it does not request a file path or human-readable class name.